In [0]:
# ============================================================
# CELL 1: Imports and Configuration
# ============================================================
import uuid
import traceback
from datetime import datetime
from pyspark.sql import functions as F

CATALOG        = "ubuntu_bank_fraud"
CONTROL_SCHEMA = "control"
VOLUME_PATH    = f"/Volumes/{CATALOG}/default/landing"

BATCH_ID  = str(uuid.uuid4())
RUN_START = datetime.utcnow()

print(f"Batch ID  : {BATCH_ID}")
print(f"Run Start : {RUN_START}")
print(f"Catalog   : {CATALOG}")
print(f"Volume    : {VOLUME_PATH}")

Batch ID  : 65e42ae8-6ab5-4594-9e18-14ad90ed43ab
Run Start : 2026-06-22 21:23:13.167439
Catalog   : ubuntu_bank_fraud
Volume    : /Volumes/ubuntu_bank_fraud/default/landing


/home/spark-5edbda3d-b6d3-465c-8e5b-0f/.ipykernel/1984/command-5581388565412942-421574715:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_START = datetime.utcnow()


In [0]:
# ============================================================
# CELL 2: Read active metadata rows
# ============================================================
metadata_df = spark.sql(f"""
    SELECT
        config_id,
        source_file_name,
        target_table,
        schema_name,
        delimiter,
        file_format,
        load_strategy,
        active_flag
    FROM {CATALOG}.{CONTROL_SCHEMA}.ctrl_ingestion_config
    WHERE active_flag = 1
    ORDER BY config_id
""")

metadata_rows = metadata_df.collect()
print(f"Tables to process: {len(metadata_rows)}")
for row in metadata_rows:
    print(f"  -> {row.source_file_name}  =>  {CATALOG}.{row.schema_name}.{row.target_table}")

Tables to process: 8
  -> customers.csv  =>  ubuntu_bank_fraud.bronze.brz_customers
  -> accounts.csv  =>  ubuntu_bank_fraud.bronze.brz_accounts
  -> cards.csv  =>  ubuntu_bank_fraud.bronze.brz_cards
  -> merchants.csv  =>  ubuntu_bank_fraud.bronze.brz_merchants
  -> devices.csv  =>  ubuntu_bank_fraud.bronze.brz_devices
  -> customer_device_link.csv  =>  ubuntu_bank_fraud.bronze.brz_customer_device_link
  -> transactions.csv  =>  ubuntu_bank_fraud.bronze.brz_transactions
  -> date_dim.csv  =>  ubuntu_bank_fraud.bronze.brz_date_dim


In [0]:
# ============================================================
# CELL 3: Reusable audit writer
# ============================================================
def write_audit(batch_id, table_name, start_time, end_time,
                rows_loaded, status, error_message=None):

    audit_row = [(
        batch_id,
        table_name,
        start_time,
        end_time,
        rows_loaded,
        status,
        error_message or ""
    )]

    audit_df = spark.createDataFrame(
        audit_row,
        schema=["batch_id", "table_name", "start_time",
                "end_time", "rows_loaded", "status", "error_message"]
    )

    (
        audit_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(f"{CATALOG}.{CONTROL_SCHEMA}.ctrl_pipeline_audit")
    )

print("Audit writer function registered.")

Audit writer function registered.


In [0]:
# ============================================================
# CELL 4: Data Quality validation functions
# ============================================================
def check_row_count(df, table_name, min_rows=1):
    count = df.count()
    if count < min_rows:
        raise ValueError(f"[DQ FAIL] {table_name}: row count {count} < minimum {min_rows}")
    print(f"  [DQ] Row count  : {count:,}  PASS")
    return count


def check_nulls(df, table_name):
    total = df.count()
    for col in df.columns:
        null_count = df.filter(F.col(col).isNull()).count()
        pct = (null_count / total * 100) if total > 0 else 0
        if pct > 10:
            print(f"  [DQ WARN] {table_name}.{col}: {pct:.1f}% nulls")
    print(f"  [DQ] Null check : complete")


def check_schema(df, table_name):
    if len(df.columns) == 0:
        raise ValueError(f"[DQ FAIL] {table_name}: schema inference returned 0 columns")
    print(f"  [DQ] Schema     : {len(df.columns)} columns  PASS")


def check_duplicates(df, table_name):
    total    = df.count()
    distinct = df.distinct().count()
    dupes    = total - distinct
    if dupes > 0:
        print(f"  [DQ WARN] {table_name}: {dupes:,} fully duplicate rows")
    else:
        print(f"  [DQ] Duplicates : 0  PASS")


def run_all_dq(df, table_name):
    check_schema(df, table_name)
    row_count = check_row_count(df, table_name)
    check_nulls(df, table_name)
    check_duplicates(df, table_name)
    return row_count

print("DQ functions registered.")

DQ functions registered.


In [0]:
# ============================================================
# CELL 5: Dynamic ingestion loop — metadata-driven
# ============================================================
results = []

for row in metadata_rows:

    table_start = datetime.utcnow()
    full_table  = f"{CATALOG}.{row.schema_name}.{row.target_table}"
    file_path   = f"{VOLUME_PATH}/{row.source_file_name}"

    print(f"\n[START] {row.target_table}")
    print(f"  Source : {file_path}")
    print(f"  Target : {full_table}")

    try:
        # 1. Read source file dynamically
        raw_df = (
            spark.read
                .format(row.file_format)
                .option("header",      "true")
                .option("inferSchema", "true")
                .option("sep",         row.delimiter)
                .option("multiLine",   "true")
                .option("escape",      "\"")
                .load(file_path)
        )

        # 2. Add framework audit columns
        enriched_df = (
            raw_df
                .withColumn("_load_ts",     F.current_timestamp())
                .withColumn("_batch_id",    F.lit(BATCH_ID))
                .withColumn("_source_file", F.lit(row.source_file_name))
        )

        # 3. Run Data Quality checks
        row_count = run_all_dq(enriched_df, row.target_table)

        # 4. Write to Delta table
        (
            enriched_df.write
                .format("delta")
                .mode(row.load_strategy.lower())
                .option("mergeSchema", "true")
                .saveAsTable(full_table)
        )

        # 5. Verify row count after write
        written = spark.table(full_table).count()
        print(f"  [OK] Written : {written:,} rows -> {full_table}")

        table_end = datetime.utcnow()
        write_audit(BATCH_ID, row.target_table, table_start, table_end, written, "SUCCESS")
        results.append({"table": row.target_table, "status": "SUCCESS", "rows": written})

    except Exception as e:
        table_end = datetime.utcnow()
        err_msg   = traceback.format_exc()[-2000:]
        print(f"  [FAIL] {row.target_table}: {e}")
        write_audit(BATCH_ID, row.target_table, table_start, table_end, 0, "FAILED", err_msg)
        results.append({"table": row.target_table, "status": "FAILED", "rows": 0})

/home/spark-5edbda3d-b6d3-465c-8e5b-0f/.ipykernel/1984/command-5581388565412938-3351455735:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  table_start = datetime.utcnow()



[START] brz_customers
  Source : /Volumes/ubuntu_bank_fraud/default/landing/customers.csv
  Target : ubuntu_bank_fraud.bronze.brz_customers
  [DQ] Schema     : 20 columns  PASS
  [DQ] Row count  : 30,000  PASS
  [DQ] Null check : complete
  [DQ] Duplicates : 0  PASS
  [OK] Written : 30,000 rows -> ubuntu_bank_fraud.bronze.brz_customers


/home/spark-5edbda3d-b6d3-465c-8e5b-0f/.ipykernel/1984/command-5581388565412938-3351455735:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  table_end = datetime.utcnow()



[START] brz_accounts
  Source : /Volumes/ubuntu_bank_fraud/default/landing/accounts.csv
  Target : ubuntu_bank_fraud.bronze.brz_accounts
  [DQ] Schema     : 18 columns  PASS
  [DQ] Row count  : 45,000  PASS
  [DQ WARN] brz_accounts.closure_date: 97.1% nulls
  [DQ] Null check : complete
  [DQ] Duplicates : 0  PASS
  [OK] Written : 45,000 rows -> ubuntu_bank_fraud.bronze.brz_accounts

[START] brz_cards
  Source : /Volumes/ubuntu_bank_fraud/default/landing/cards.csv
  Target : ubuntu_bank_fraud.bronze.brz_cards
  [DQ] Schema     : 18 columns  PASS
  [DQ] Row count  : 39,705  PASS
  [DQ WARN] brz_cards.replacement_reason: 100.0% nulls
  [DQ WARN] brz_cards.replaced_card_id: 100.0% nulls
  [DQ] Null check : complete
  [DQ] Duplicates : 0  PASS
  [OK] Written : 39,705 rows -> ubuntu_bank_fraud.bronze.brz_cards

[START] brz_merchants
  Source : /Volumes/ubuntu_bank_fraud/default/landing/merchants.csv
  Target : ubuntu_bank_fraud.bronze.brz_merchants
  [DQ] Schema     : 22 columns  PASS
  [DQ

In [0]:
# ============================================================
# CELL 6: Batch summary
# ============================================================
success = [r for r in results if r["status"] == "SUCCESS"]
failed  = [r for r in results if r["status"] == "FAILED"]

print(f"\n{'='*54}")
print(f"BATCH SUMMARY  |  Batch ID: {BATCH_ID[:8]}...")
print(f"{'='*54}")
print(f"  Tables processed : {len(results)}")
print(f"  Success          : {len(success)}")
print(f"  Failed           : {len(failed)}")
print(f"  Total rows       : {sum(r['rows'] for r in success):,}")
print(f"{'='*54}")

for r in results:
    icon = "✓" if r["status"] == "SUCCESS" else "✗"
    print(f"  {icon}  {r['table']:<35}  {r['rows']:>10,} rows")

if failed:
    raise Exception(f"{len(failed)} table(s) failed. Check audit table for details.")


BATCH SUMMARY  |  Batch ID: 65e42ae8...
  Tables processed : 8
  Success          : 8
  Failed           : 0
  Total rows       : 2,380,342
  ✓  brz_customers                            30,000 rows
  ✓  brz_accounts                             45,000 rows
  ✓  brz_cards                                39,705 rows
  ✓  brz_merchants                             2,200 rows
  ✓  brz_devices                              28,000 rows
  ✓  brz_customer_device_link                 34,706 rows
  ✓  brz_transactions                      2,200,000 rows
  ✓  brz_date_dim                                731 rows
